In [1]:
!pip install pymilvus sentence-transformers transformers chromadb pandas tqdm pyarrow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.8/386.8 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
# !pip install "pymilvus[milvus_lite]" sentence-transformers transformers chromadb pandas tqdm pyarrow -q

In [3]:
!pip install -i https://mirrors.aliyun.com/pypi/simple/ pytorch-crf==0.7.2

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


In [4]:
import os
import glob
import json
import time
import warnings
import gc
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import autocast
from google.colab import drive
from transformers import AutoTokenizer, AutoConfig, AutoModel
from transformers.modeling_outputs import TokenClassifierOutput, SequenceClassifierOutput
from torchcrf import CRF
from tqdm import tqdm

warnings.filterwarnings("ignore")

In [5]:
# ==================== دستگاه ====================
from google.colab import drive

# اول سعی می‌کنیم unmount کنیم (اگه یه mount نیمه‌کاره باشه)
try:
    drive.flush_and_unmount()
    print("✅ unmount قبلی انجام شد")
except Exception as e:
    print("چیزی برای unmount نبود:", e)

# حالا با force_remount دوباره mount می‌کنیم
drive.mount('/content/drive', force_remount=True)

Drive not mounted, so nothing to flush and unmount.
✅ unmount قبلی انجام شد
Mounted at /content/drive


In [15]:
# ==================== تنظیمات ====================
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks"
DATA_DIR = os.path.join(BASE_DIR, "Cleaned_output")
COMMENTS_DIR = os.path.join(DATA_DIR, "digikala-comments_parts")
OUTPUT_DIR = os.path.join(DATA_DIR, "absa_results_comments")
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOCAL_MODELS_DIR = "/content/drive/MyDrive/Colab Notebooks/pars_absa_models"
sentiment_path = os.path.join(LOCAL_MODELS_DIR, "sentiment_model")
aspect_path = os.path.join(LOCAL_MODELS_DIR, "aspect_model")

MODEL_NAME = "HooshvareLab/bert-base-parsbert-uncased"
MAX_LENGTH = 160
CHUNK_SIZE = 4000            # هرچه بزرگتر، مرتب‌سازی طول موثرتر (padding کمتر) - اگر RAM اجازه داد بیشتر کنید
ASPECT_MINI_BATCH = 128      # بسته به حافظه‌ی GPU؛ اگر OOM نگرفتید امتحان کنید 192/256
SENTIMENT_MINI_BATCH = 256
USE_AMP = torch.cuda.is_available()   # اجرای mixed precision (fp16) روی GPU

TEXT_COLUMN_CANDIDATES = ["raw_text_normalized", "raw_text"]  # طبق schema شما - فقط همین خوانده می‌شود

label2id = {"negative": 0, "neutral": 1, "positive": 2}
id2label = {v: k for k, v in label2id.items()}
aspect_label2id = {"O": 0, "B-ASP": 1, "I-ASP": 2}
aspect_id2label = {v: k for k, v in aspect_label2id.items()}


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 دستگاه: {device.upper()} | AMP فعال: {USE_AMP}")

🚀 دستگاه: CUDA | AMP فعال: True


In [16]:
# ==================== نرمال‌سازی aspect (حذف پسوندهای ملکی/جمع) ====================
# _ASPECT_SUFFIXES = ["های","های", "شون","مون" ,"مان","تون","تان", "شون","شان", "اش", "ها", "ش", "ی"]

# def _strip_one_suffix(word):
#     for suf in _ASPECT_SUFFIXES:
#         if word.endswith(suf) and len(word) - len(suf) >= 2:
#             return word[: -len(suf)], True
#     return word, False

def normalize_aspect_term(term):
    term = term.strip().replace("‌", "")
    if not term:
        return term
    words = term.split()
    if not words:
        return term
    last = words[-1]
    # for _ in range(2):
    #     new_last, changed = _strip_one_suffix(last)
    #     if not changed:
    #         break
    #     last = new_last
    # words[-1] = last
    return " ".join(words).strip()

# ==================== کلاس‌های اختصاصی مدل ====================
class BertBiLSTMCrfForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id,
                 use_crf=True, use_bilstm=True, dropout_p=0.2):
        super().__init__()
        self.num_labels = num_labels
        self.use_crf = use_crf
        self.use_bilstm = use_bilstm
        self.config = AutoConfig.from_pretrained(model_name, num_labels=num_labels,
                                                 id2label=id2label, label2id=label2id)
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout_p)
        hidden = self.config.hidden_size
        if use_bilstm:
            self.bilstm = nn.LSTM(hidden, hidden // 2, batch_first=True, bidirectional=True, dropout=0.1)
        else:
            self.bilstm = None
        self.classifier = nn.Linear(hidden, num_labels)
        self.crf = CRF(num_labels, batch_first=True) if use_crf else None

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        if self.use_bilstm and self.bilstm is not None:
            sequence_output, _ = self.bilstm(sequence_output)
        emissions = self.classifier(sequence_output)
        mask = attention_mask.bool()
        if self.use_crf:
            # CRF.decode با ورودی float16 مشکل دارد - قبل از decode به float32 برمی‌گردانیم
            decoded = self.crf.decode(emissions.float(), mask=mask)
            pred_ids = torch.full(emissions.shape[:2], -100, dtype=torch.long, device=emissions.device)
            for i, seq in enumerate(decoded):
                pred_ids[i, :len(seq)] = torch.tensor(seq, device=emissions.device)
            logits_out = pred_ids
        else:
            logits_out = torch.argmax(emissions, dim=-1)
        return TokenClassifierOutput(logits=logits_out)


class BertForWeightedSentimentClassification(nn.Module):
    def __init__(self, model_name, num_labels, id2label, label2id, dropout_p=0.2):
        super().__init__()
        self.num_labels = num_labels
        self.config = AutoConfig.from_pretrained(
            model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
        )
        self.bert = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout_p)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs[0][:, 0, :]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        return SequenceClassifierOutput(logits=logits)

# ==================== بارگذاری مدل‌ها ====================
aspect_tokenizer = AutoTokenizer.from_pretrained(aspect_path, use_fast=True)
sentiment_tokenizer = AutoTokenizer.from_pretrained(sentiment_path, use_fast=True)

aspect_model = BertBiLSTMCrfForTokenClassification(
    MODEL_NAME, num_labels=3, id2label=aspect_id2label, label2id=aspect_label2id,
    use_crf=True, use_bilstm=True
)
aspect_state = torch.load(os.path.join(aspect_path, "pytorch_model.bin"), map_location=device)
m, u = aspect_model.load_state_dict(aspect_state, strict=False)
print("Aspect model -> missing:", m, "| unexpected:", u)
aspect_model.to(device).eval()

sentiment_model = BertForWeightedSentimentClassification(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)
sentiment_state = torch.load(os.path.join(sentiment_path, "pytorch_model.bin"), map_location=device)
m, u = sentiment_model.load_state_dict(sentiment_state, strict=False)
print("Sentiment model -> missing:", m, "| unexpected:", u)
sentiment_model.to(device).eval()

print("✅ هر دو مدل بارگذاری شدند.")

# ==================== استخراج span های aspect (با مرتب‌سازی طول + AMP) ====================
@torch.inference_mode()
def extract_aspect_spans(texts, mini_batch_size=ASPECT_MINI_BATCH):
    n = len(texts)
    results = [None] * n

    # مرتب‌سازی بر اساس طول متن تا padding داخل هر batch حداقل شود
    order = sorted(range(n), key=lambda i: len(texts[i]))

    for start in range(0, n, mini_batch_size):
        batch_idx = order[start:start + mini_batch_size]
        sub_texts = [texts[i] for i in batch_idx]

        enc = aspect_tokenizer(
            sub_texts, truncation=True, padding=True, max_length=MAX_LENGTH,
            return_tensors="pt", return_offsets_mapping=True
        )
        offsets_batch = enc.pop("offset_mapping")
        enc = {k: v.to(device) for k, v in enc.items()}

        with autocast(enabled=USE_AMP):
            out = aspect_model(**enc)
        pred_ids = out.logits.cpu()

        for b, orig_i in enumerate(batch_idx):
            text = texts[orig_i]
            offsets = offsets_batch[b].tolist()
            tags = pred_ids[b].tolist()
            spans = []
            cur_start = cur_end = None
            for (tok_start, tok_end), tag in zip(offsets, tags):
                if tag == -100 or tok_start == tok_end:
                    if cur_start is not None:
                        spans.append((cur_start, cur_end))
                        cur_start = None
                    continue
                tag_name = aspect_id2label.get(int(tag), "O")
                if tag_name == "B-ASP":
                    if cur_start is not None:
                        spans.append((cur_start, cur_end))
                    cur_start, cur_end = tok_start, tok_end
                elif tag_name == "I-ASP" and cur_start is not None:
                    cur_end = tok_end
                else:
                    if cur_start is not None:
                        spans.append((cur_start, cur_end))
                        cur_start = None
            if cur_start is not None:
                spans.append((cur_start, cur_end))

            span_terms = [(s, e, text[s:e].strip()) for s, e in spans if e > s and text[s:e].strip()]
            results[orig_i] = span_terms

    return results

# ==================== طبقه‌بندی sentiment (با مرتب‌سازی طول + AMP) ====================
@torch.inference_mode()
def classify_sentiment_pairs(pairs, mini_batch_size=SENTIMENT_MINI_BATCH):
    n = len(pairs)
    if n == 0:
        return np.zeros((0, 3), dtype=np.float32)

    all_probs = np.zeros((n, 3), dtype=np.float32)
    order = sorted(range(n), key=lambda i: len(pairs[i][0]) + len(pairs[i][1]))

    for start in range(0, n, mini_batch_size):
        batch_idx = order[start:start + mini_batch_size]
        texts_ = [pairs[i][0] for i in batch_idx]
        terms_ = [pairs[i][1] for i in batch_idx]

        enc = sentiment_tokenizer(
            texts_, terms_, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt"
        ).to(device)

        with autocast(enabled=USE_AMP):
            out = sentiment_model(**enc)
        probs = torch.softmax(out.logits.float(), dim=-1).cpu().numpy()

        for b, orig_i in enumerate(batch_idx):
            all_probs[orig_i] = probs[b]

    return all_probs

# ==================== پردازش یک chunk ====================
def process_chunk(texts):
    span_lists = extract_aspect_spans(texts)

    flat_pairs, owner_idx, raw_terms = [], [], []
    for c_idx, spans in enumerate(span_lists):
        for s, e, raw_term in spans:
            flat_pairs.append((texts[c_idx], raw_term))
            owner_idx.append(c_idx)
            raw_terms.append(raw_term)

    probs = classify_sentiment_pairs(flat_pairs)

    per_comment_agg = [dict() for _ in texts]
    for i, (c_idx, raw_term) in enumerate(zip(owner_idx, raw_terms)):
        norm_term = normalize_aspect_term(raw_term)
        if not norm_term:
            continue
        per_comment_agg[c_idx].setdefault(norm_term, []).append(probs[i])

    results = []
    for c_idx in range(len(texts)):
        aspects_out = []
        for norm_term, prob_list in per_comment_agg[c_idx].items():
            avg_prob = np.mean(prob_list, axis=0)
            pred_id = int(np.argmax(avg_prob))
            aspects_out.append({
                "term": norm_term,
                "sentiment": id2label[pred_id],
                "negative_pct": round(float(avg_prob[0]) * 100, 1),
                "neutral_pct": round(float(avg_prob[1]) * 100, 1),
                "positive_pct": round(float(avg_prob[2]) * 100, 1),
                "mentions": len(prob_list),
            })
        results.append(aspects_out)
    return results

# ==================== حلقه‌ی اصلی ====================
def get_review_text_series(df):
    for col in TEXT_COLUMN_CANDIDATES:
        if col in df.columns:
            return df[col].fillna("").astype(str), col
    raise ValueError(f"هیچ‌کدام از ستون‌های {TEXT_COLUMN_CANDIDATES} در فایل پیدا نشد.")

def run():
    parquet_files = glob.glob(os.path.join(COMMENTS_DIR, "**/*.parquet"), recursive=True)
    print(f"🔎 فایل‌های کامنت پیدا شده: {len(parquet_files)} در مسیر {COMMENTS_DIR}")
    if not parquet_files:
        print("⚠️ هیچ فایلی پیدا نشد.")
        return

    files_done, files_skipped, rows_total = 0, 0, 0
    overall_start = time.time()

    for parq_file in parquet_files:
        out_name = "absa_" + os.path.splitext(os.path.basename(parq_file))[0] + ".parquet"
        out_path = os.path.join(OUTPUT_DIR, out_name)

        if os.path.exists(out_path):
            files_skipped += 1
            print(f"⏭️ {os.path.basename(parq_file)} قبلاً کامل پردازش شده - رد شد.")
            continue

        df = pd.read_parquet(parq_file)
        review_texts, used_col = get_review_text_series(df)
        print(f"📂 {os.path.basename(parq_file)}: {len(df):,} ردیف | ستون متن: '{used_col}'")

        file_start = time.time()
        all_rows = []

        for i in tqdm(range(0, len(df), CHUNK_SIZE), desc=os.path.basename(parq_file)):
            batch_records = df.iloc[i:i + CHUNK_SIZE].to_dict('records')  # همه‌ی ستون‌های اصلی حفظ می‌شوند
            texts = review_texts.iloc[i:i + CHUNK_SIZE].tolist()

            chunk_results = process_chunk(texts)

            for idx, record in enumerate(batch_records):
                record["aspects_json"] = json.dumps(chunk_results[idx], ensure_ascii=False)
                record["num_aspects"] = len(chunk_results[idx])
                all_rows.append(record)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        tmp_path = out_path + ".tmp"
        pd.DataFrame(all_rows).to_parquet(tmp_path, index=False)
        os.rename(tmp_path, out_path)

        elapsed = time.time() - file_start
        files_done += 1
        rows_total += len(all_rows)
        print(f"💾 ذخیره شد: {out_path} ({len(all_rows):,} ردیف) در {elapsed:.1f} ثانیه "
              f"({len(all_rows)/max(elapsed,0.01):.1f} کامنت/ثانیه)")

    total_elapsed = time.time() - overall_start
    print(f"\n✅ تمام شد! فایل‌های جدید: {files_done} | رد‌شده: {files_skipped} | مجموع کامنت‌ها: {rows_total:,}")
    print(f"⏱️ کل زمان: {total_elapsed/3600:.2f} ساعت")
    print(f"📁 خروجی‌ها در: {OUTPUT_DIR}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Aspect model -> missing: [] | unexpected: []


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: HooshvareLab/bert-base-parsbert-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentiment model -> missing: [] | unexpected: []
✅ هر دو مدل بارگذاری شدند.


In [17]:
if __name__ == "__main__":
    run()

🔎 فایل‌های کامنت پیدا شده: 124 در مسیر /content/drive/MyDrive/Colab Notebooks/Cleaned_output/digikala-comments_parts
📂 part_0002.parquet: 50,000 ردیف | ستون متن: 'raw_text_normalized'


part_0002.parquet: 100%|██████████| 13/13 [00:53<00:00,  4.11s/it]


💾 ذخیره شد: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments/absa_part_0002.parquet (50,000 ردیف) در 54.0 ثانیه (925.7 کامنت/ثانیه)
📂 part_0001.parquet: 50,000 ردیف | ستون متن: 'raw_text_normalized'


part_0001.parquet: 100%|██████████| 13/13 [00:54<00:00,  4.17s/it]


💾 ذخیره شد: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments/absa_part_0001.parquet (50,000 ردیف) در 54.9 ثانیه (910.9 کامنت/ثانیه)
📂 part_0004.parquet: 50,000 ردیف | ستون متن: 'raw_text_normalized'


...

💾 ذخیره شد: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments/absa_part_0123.parquet (3,060 ردیف) در 5.7 ثانیه (538.7 کامنت/ثانیه)
📂 part_0122.parquet: 50,000 ردیف | ستون متن: 'raw_text_normalized'


part_0122.parquet: 100%|██████████| 13/13 [00:59<00:00,  4.60s/it]


💾 ذخیره شد: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments/absa_part_0122.parquet (50,000 ردیف) در 60.4 ثانیه (828.1 کامنت/ثانیه)
📂 part_0121.parquet: 50,000 ردیف | ستون متن: 'raw_text_normalized'


part_0121.parquet: 100%|██████████| 13/13 [01:05<00:00,  5.07s/it]


💾 ذخیره شد: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments/absa_part_0121.parquet (50,000 ردیف) در 66.6 ثانیه (751.0 کامنت/ثانیه)

✅ تمام شد! فایل‌های جدید: 124 | رد‌شده: 0 | مجموع کامنت‌ها: 6,153,060
⏱️ کل زمان: 1.87 ساعت
📁 خروجی‌ها در: /content/drive/MyDrive/Colab Notebooks/Cleaned_output/absa_results_comments
